In [1]:
import pandas as pd

df = pd.read_csv("02) IMDB Dataset.csv")

In [2]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [3]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [4]:
df.drop_duplicates(inplace = True)

In [5]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


1. converting to lower case 

In [6]:
df["review"] = df["review"].str.lower()

2. Removing the URLS

In [7]:
import re

def remove_urls(text):
    text = re.sub(r"http\S+" , "" , text)
    return text

df["review"] = df["review"].apply(remove_urls)

In [8]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]" ,"",text)
    return text

df["review"] = df["review"].apply(remove_punctuations)

In [9]:
def remove_html(text):
    text = re.sub(r"<.*?>","",text)
    return text

df["review"] = df["review"].apply(remove_html)

In [10]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\himan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\himan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\himan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [11]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

!pip install pandarallel
from pandarallel import pandarallel

# Initialize pandarallel (it will automatically detect your CPU cores)
pandarallel.initialize(progress_bar=True)

# Replace standard .apply() with .parallel_apply()


INFO: Pandarallel will run on 8 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.

https://nalepae.github.io/pandarallel/troubleshooting/


In [12]:
def remove_stopwords(text):

    from nltk.tokenize import word_tokenize
    from nltk.corpus import stopwords
    
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word,"")

    return text

df["review"] = df["review"].parallel_apply(remove_stopwords)


In [13]:
def stemming(text):
    

    from nltk.tokenize import word_tokenize
    from nltk.stem import PorterStemmer
    
    ps = PorterStemmer()
    stemmed_words = []
    
    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].parallel_apply(stemming)


In [14]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features= 5000)

X = tf.fit_transform(df["review"])

In [16]:
from sklearn.model_selection import train_test_split
y = df["sentiment"]
X_train , X_test , Y_train , Y_test = train_test_split(
    X , y , test_size = 0.2 , random_state = 42
)


In [17]:
import torch

from torch.utils.data import TensorDataset , DataLoader

X_train = X_train.toarray()
X_test = X_test.toarray()

train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()                 
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

NameError: name 'y_train' is not defined

In [ ]:
train_loader = DataLoader(train_set, shuffle = True,batch_size = 64)
test_loader = DataLoader(test_set, shuffle = True,batch_size = 64)